# Evaluate a real classifier

This code is an example of the use of VADER classifier from NLTK. It is a Naive-Bayes classifier that is trainded with a lexicon and dataset of movie reviews.

Look in the example how the library SKLearn is used to evaulate the classifier.

At the end you have an example on how to use the classifier en custom examples. 


In [1]:

import nltk
from nltk.corpus import movie_reviews
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report, confusion_matrix
import random

# Download required NLTK datasets
nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')

# Preprocess the data
stop_words = set(stopwords.words('english'))

def extract_features(words):
    return {word: True for word in words if word.lower() not in stop_words}

# Prepare the dataset
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]
random.shuffle(documents)  # Shuffle the dataset for better randomness

# Feature extraction
feature_sets = [(extract_features(words), category) for (words, category) in documents]

# Split the data into training and testing sets
train_size = int(len(feature_sets) * 0.8)
train_set, test_set = feature_sets[:train_size], feature_sets[train_size:]

# Train a Naive Bayes Classifier
classifier = NaiveBayesClassifier.train(train_set)

# Evaluate the classifier
print("\nNaive Bayes Classifier Evaluation:")
print(f"Accuracy: {accuracy(classifier, test_set) * 100:.2f}%")
classifier.show_most_informative_features(10)

# Prepare predictions and true labels for sklearn metrics
y_true = [label for (_, label) in test_set]
y_pred = [classifier.classify(features) for (features, _) in test_set]

# Evaluate using sklearn metrics
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# VADER Sentiment Analysis on custom examples
sia = SentimentIntensityAnalyzer()
example_sentences = [
    "I absolutely loved this movie! The acting was fantastic.",
    "This was the worst film I have ever seen.",
    "The plot was predictable, but the cinematography was beautiful.",
    "I wouldn't recommend it. It was boring and too long."
]

print("\nVADER Sentiment Analysis:")
for sentence in example_sentences:
    score = sia.polarity_scores(sentence)
    sentiment = "positive" if score['compound'] > 0 else "negative"
    print(f"Sentence: {sentence}\nSentiment: {sentiment} (Score: {score['compound']})\n")


[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/bernardoquindimil/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/bernardoquindimil/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bernardoquindimil/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/bernardoquindimil/nltk_data...



Naive Bayes Classifier Evaluation:
Accuracy: 73.25%
Most Informative Features
             outstanding = True              pos : neg    =     14.2 : 1.0
                   sucks = True              neg : pos    =     12.3 : 1.0
                  tucker = True              pos : neg    =     11.7 : 1.0
                  avoids = True              pos : neg    =     11.1 : 1.0
             fascination = True              pos : neg    =     10.4 : 1.0
                seamless = True              pos : neg    =     10.4 : 1.0
                  hudson = True              neg : pos    =     10.3 : 1.0
           unintentional = True              neg : pos    =     10.3 : 1.0
              accessible = True              pos : neg    =      9.7 : 1.0
              astounding = True              pos : neg    =      9.7 : 1.0

Classification Report:
              precision    recall  f1-score   support

         neg       0.94      0.49      0.64       198
         pos       0.66      0.97     

# Exercise:

Create your own gold standard and measure Precission, Recall, and F1 manually and with SKLearn to check if the result is the same. 

In [4]:
from sklearn.metrics import precision_score, recall_score, f1_score

# This is an example
# Create the gold standard
true_labels = [1, 0, 1, 1, 0, 1, 0, 0, 1, 0]

# Model predicted labels
predicted_labels = [1, 0, 1, 0, 0, 1, 0, 1, 1, 0]

# Calculate the metrics doing a confusion matrix
TP = FP = TN = FN = 0
for true, pred in zip(true_labels, predicted_labels):
    if true == 1 and pred == 1:
        TP += 1
    elif true == 0 and pred == 1:
        FP += 1
    elif true == 0 and pred == 0:
        TN += 1
    elif true == 1 and pred == 0:
        FN += 1

precision_manual = TP / (TP + FP) if (TP + FP) > 0 else 0   # else 0 because if TP + FP is 0 it can't divide
recall_manual = TP / (TP + FN) if (TP + FN) > 0 else 0
f1_manual = 2 * (precision_manual * recall_manual) / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print(f"Precisión manual: {precision_manual:.2f}")
print(f"Recall manual: {recall_manual:.2f}")
print(f"F1 Score manual: {f1_manual:.2f}")

# Paso 3: Calculate the metrics with sklearn
precision_sklearn = precision_score(true_labels, predicted_labels)
recall_sklearn = recall_score(true_labels, predicted_labels)
f1_sklearn = f1_score(true_labels, predicted_labels)

print(f"Precisión con SKLearn: {precision_sklearn:.2f}")
print(f"Recall con SKLearn: {recall_sklearn:.2f}")
print(f"F1 Score con SKLearn: {f1_sklearn:.2f}")

Precisión manual: 0.80
Recall manual: 0.80
F1 Score manual: 0.80
Precisión con SKLearn: 0.80
Recall con SKLearn: 0.80
F1 Score con SKLearn: 0.80


Yes, the result is the same